# CNN 2D — Music Genre Classification

Trains a 2D Convolutional Neural Network on mel-spectrogram tensors for music genre classification.

## Prerequisites

**Dataset:** Add the preprocessed Kaggle dataset to this notebook.
`TENSORS_DIR` must point to `/kaggle/input/pr-a2-preprocesseddataset`.

**Repository:** The repo must be cloned into `/kaggle/working/` so that `src/` modules are accessible:

```bash
git clone https://github.com/marticasasn/ELEC3612-Assignment2
```

Run the clone cell below before executing any other cell.

In [ ]:
from pathlib import Path
REPO_DIR = Path("/kaggle/working/ELEC3612-Assignment2")
if not REPO_DIR.exists():
    !git clone https://github.com/marticasasn/ELEC3612-Assignment2 {REPO_DIR}
else:
    print("Repository already exists, skipping clone.")

## 1. Imports and Configuration

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

sys.path.insert(0, str(REPO_DIR))

from src.dataset import SpectrogramDataset
from src.evaluate import evaluate_model
from src.models import CNNClassifier
from src.train import train

# --- Paths ---
TENSORS_DIR     = Path("/kaggle/input/pr-a2-preprocesseddataset")
CHECKPOINT_PATH = Path("/kaggle/working/best_cnn.pth")
FIGURES_DIR     = Path("/kaggle/working/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# --- Device ---
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"

# --- Hyperparameters ---
BATCH_SIZE    = 64      # TUNE
NUM_EPOCHS    = 50      # TUNE
PATIENCE      = 10      # TUNE
LEARNING_RATE = 1e-3    # TUNE
WEIGHT_DECAY  = 1e-4    # TUNE

print(f"Device: {DEVICE}")

## 1.1 Reproducibility

In [ ]:
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
print(f"Random seed set to {SEED}")

## 2. Load Datasets

Load pre-extracted mel-spectrogram tensors for each split and wrap them in DataLoaders.

In [ ]:
from torch.utils.data import DataLoader

train_dataset = SpectrogramDataset(TENSORS_DIR, split="train")
val_dataset   = SpectrogramDataset(TENSORS_DIR, split="val")
test_dataset  = SpectrogramDataset(TENSORS_DIR, split="test")

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=PIN_MEMORY,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=PIN_MEMORY,
)

print(f"Train size: {len(train_dataset):,}")
print(f"Val size:   {len(val_dataset):,}")
print(f"Test size:  {len(test_dataset):,}")
print(f"Num classes: {train_dataset.num_classes}")
print(f"Genres: {train_dataset.genres}")

class_weights = train_dataset.get_class_weights()
print(f"\nClass weights: {class_weights}")

## 3. Model Setup

Instantiate the CNN, loss function, optimiser, and learning rate scheduler.

In [ ]:
model = CNNClassifier(num_classes=train_dataset.num_classes).to(DEVICE)

print(model.summary())
print(f"Trainable parameters: {model.count_parameters():,}")

criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))

optimizer = optim.Adam(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=5, factor=0.5
)

## 4. Training

Train for up to `NUM_EPOCHS` epochs with early stopping on validation accuracy.
Best checkpoint is saved to `CHECKPOINT_PATH`.

In [ ]:
history = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=DEVICE,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    checkpoint_path=CHECKPOINT_PATH,
    scheduler=scheduler,
)

print(f"\nBest val accuracy: {history['best_val_acc']:.4f}")
print(f"Best epoch:        {history['best_epoch']}")

## 5. Evaluation on Test Set

Load the best checkpoint and evaluate on the held-out test set.
Saves confusion matrix and training curves to `FIGURES_DIR`.

In [ ]:
metrics = evaluate_model(
    model=model,
    loader=test_loader,
    device=DEVICE,
    class_names=test_dataset.genres,
    checkpoint_path=CHECKPOINT_PATH,
    figures_dir=FIGURES_DIR,
    model_name="cnn",
)

print(f"Test accuracy:    {metrics['accuracy']:.4f}")
print(f"Weighted F1:      {metrics['weighted_f1']:.4f}")
print(f"Macro F1:         {metrics['macro_f1']:.4f}")

## 6. Save Results

Save metrics, per-sample predictions, and list all output files.

In [ ]:
import pandas as pd
from src.evaluate import get_predictions

# Save metrics
metrics_path = Path("/kaggle/working/cnn_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to {metrics_path}")

# Save per-sample predictions
y_true, y_pred = get_predictions(model, test_loader, DEVICE)
predictions_path = Path("/kaggle/working/cnn_predictions.csv")
pd.DataFrame({
    "y_true": y_true,
    "y_pred": y_pred,
    "true_genre": [test_dataset.genres[i] for i in y_true],
    "pred_genre": [test_dataset.genres[i] for i in y_pred],
}).to_csv(predictions_path, index=False)
print(f"Predictions saved to {predictions_path}")

# List output files
output_files = [
    CHECKPOINT_PATH,
    metrics_path,
    predictions_path,
    *sorted(FIGURES_DIR.glob("cnn_*.png")),
]
print("\nOutput files:")
for p in output_files:
    size_kb = p.stat().st_size / 1024 if p.exists() else 0
    print(f"  {p}  ({size_kb:.1f} KB)")